Wan 2.2 Animate (person replacement) in ComfyUI on Kaggle T4. Code + inputs come from the GitHub repo (`assets/person.jpg`, `assets/video.mp4`).

In [ ]:
!nvidia-smi -L
!df -h /kaggle/working /tmp | tail -2
!git clone --depth 1 https://github.com/bugrayildiz1/ai-clone-demo.git /kaggle/working/repo
!ls /kaggle/working/repo/assets

Install ComfyUI + nodes.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -q -r requirements.txt
%cd custom_nodes
!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF
!git clone --depth 1 https://github.com/kijai/ComfyUI-KJNodes
!git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux
!pip install -q -r ComfyUI-GGUF/requirements.txt -r ComfyUI-KJNodes/requirements.txt -r comfyui_controlnet_aux/requirements.txt
%cd ..

Models go to /tmp (large disk), symlinked in.

In [ ]:
import os, subprocess
M="/tmp/models"
files={
 "diffusion_models/Wan2.2-Animate-14B-Q4_K_S.gguf":"https://huggingface.co/QuantStack/Wan2.2-Animate-14B-GGUF/resolve/main/Wan2.2-Animate-14B-Q4_K_S.gguf",
 "text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":"https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
 "clip_vision/clip_vision_h.safetensors":"https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
 "vae/wan_2.1_vae.safetensors":"https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
 "loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors":"https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
}
for rel,url in files.items():
    dst=f"{M}/{rel}"; os.makedirs(os.path.dirname(dst),exist_ok=True)
    if not os.path.exists(dst): subprocess.run(["curl","-L","--fail","-o",dst,url],check=True)
    link=f"/kaggle/working/ComfyUI/models/{rel}"
    os.makedirs(os.path.dirname(link),exist_ok=True)
    if not os.path.lexists(link): os.symlink(dst,link)
!du -sh /tmp/models

Prep inputs: first 5 s, 480x832, 16 fps (Wan native rate).

In [ ]:
!cp /kaggle/working/repo/assets/person.jpg /kaggle/working/ComfyUI/input/person.jpg
!ffmpeg -y -loglevel error -i /kaggle/working/repo/assets/video.mp4 -t 5 -vf "scale=480:832,fps=16" -an /kaggle/working/ComfyUI/input/video_5s.mp4
!curl -L -o /kaggle/working/ComfyUI/user/default/workflows/wan_animate.json --create-dirs https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_wan2_2_14B_animate.json

Start ComfyUI + public tunnel. Open the printed trycloudflare URL, load `wan_animate`, swap Load Diffusion Model for **Unet Loader (GGUF)** (Q4_K_S), pick `person.jpg` / `video_5s.mp4`, run.

In [ ]:
!curl -L -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared
import subprocess, re, time
subprocess.Popen("cd /kaggle/working/ComfyUI && CUDA_VISIBLE_DEVICES=0 python main.py --port 8188 --lowvram > /tmp/comfy.log 2>&1", shell=True)
time.sleep(40)
t=subprocess.Popen(["/tmp/cloudflared","tunnel","--url","http://127.0.0.1:8188"],stderr=subprocess.PIPE,text=True)
for line in t.stderr:
    m=re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",line)
    if m: print(m.group(0)); break
time.sleep(10**5)